In [1]:
#!pip install kagglehub
#!pip install tf-keras

In [39]:
import os, warnings
import kagglehub
import keras
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import tf_keras as k3
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from matplotlib import gridspec
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from keras.layers import Input


In [40]:
# Download latest version
path = kagglehub.dataset_download("ryanholbrook/car-or-truck")

print("Path to dataset files:", path)

# Reproducability
def set_seed(seed=31415):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed) # set a fixed value for the hash seed secret.
    os.environ['TF_DETERMINISTIC_OPS'] = '1' # environmental var used when running on gpu to enforce behaivour
set_seed(31415)

# Set Matplotlib defaults
plt.rc('figure', autolayout=True)
plt.rc('axes', labelweight='bold', labelsize='large',
       titleweight='bold', titlesize=18, titlepad=10)
plt.rc('image', cmap='magma')
plt.show()
warnings.filterwarnings('ignore') # to clean up output cells

Path to dataset files: C:\Users\Any Authorised User\.cache\kagglehub\datasets\ryanholbrook\car-or-truck\versions\1


In [41]:
# Load training and validation sets
ds_train_ = image_dataset_from_directory(
    r'C:\Users\Any Authorised User\.cache\kagglehub\datasets\ryanholbrook\car-or-truck\versions\1\train',
    labels='inferred',
    label_mode='binary',
    image_size=[128, 128],
    interpolation='nearest',
    batch_size=64,
    shuffle=True,
)
ds_valid_ = image_dataset_from_directory(
    r'C:\Users\Any Authorised User\.cache\kagglehub\datasets\ryanholbrook\car-or-truck\versions\1\valid',
    labels='inferred',
    label_mode='binary',
    image_size=[128, 128],
    interpolation='nearest',
    batch_size=64,
    shuffle=False,
)


Found 5117 files belonging to 2 classes.
Found 5051 files belonging to 2 classes.


In [42]:
# Data Pipeline
def convert_to_float(image, label):
    image = tf.image.convert_image_dtype(image, dtype=tf.float32)
    return image, label

AUTOTUNE = tf.data.experimental.AUTOTUNE
#BATCH_SIZE = 32
ds_train = (
    ds_train_.map(convert_to_float)
    #.batch(BATCH_SIZE)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)
ds_valid = (
    ds_valid_
    .map(convert_to_float)
    #.batch(BATCH_SIZE)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

In [43]:
# import VG16 module from image net to define pretrained models. This module contains large dataset of natural images to help in the CNN

In [44]:
#import kagglehub

# Download latest version
#path = kagglehub.dataset_download("ryanholbrook/cv-course-models")

#print("Path to dataset files:", path)

In [45]:
# Define the path to the folder containing 'saved_model.pb'
model_path =  r'C:\Users\Any Authorised User\.cache\kagglehub\datasets\ryanholbrook\cv-course-models\versions\4\cv-course-models\vgg16-pretrained-base'
pretrained_base = keras.layers.TFSMLayer(model_path, call_endpoint='serving_default')

pretrained_base.trainable = False

In [46]:
# check shape
for image, label in ds_train.take(1):
    print(image.shape)
    print(label.shape)

(64, 128, 128, 3)
(64, 1)


In [47]:
inputs = keras.Input(shape=(128, 128, 3))
x = pretrained_base(inputs) 
x = x['block5_pool'] 
x = layers.Flatten()(x)
x = layers.Dense(6, activation='relu')(x)
outputs = layers.Dense(1, activation='sigmoid')(x) 
model = keras.Model(inputs, outputs)

In [36]:
# let's attach the head of the classifier

#model = keras.Sequential([
  #  pretrained_base,
  #  layers.Flatten(),
  #  layers.Dense(6, activation='relu'),
  #  layers.Dense(1, activation='sigmoid')
#])

In [48]:
# train the model since this is a 2 class problem  binary versions of crossentropy and accuracy
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['binary_accuracy'],
)

In [ ]:
history = model.fit(
    ds_train,
    validation_data=ds_valid,
    epochs=30,
    verbose=1,
)

Epoch 1/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 994s 13s/step - binary_accuracy: 0.6850 - loss: 0.6037 - val_binary_accuracy: 0.7139 - val_loss: 0.5453
Epoch 2/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 572s 7s/step - binary_accuracy: 0.7887 - loss: 0.5192 - val_binary_accuracy: 0.7731 - val_loss: 0.5046
Epoch 3/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - binary_accuracy: 0.8300 - loss: 0.4824